# ✈️ AeroLex — Evidence-first answers for airline passengers

**Workshop challenge:** build a passenger-rights assistant over EU/US regulations
and Delta contracts of carriage.

> “My flight was cancelled. What refund or compensation applies?”

**Learning goal:** find relevant evidence, refine the shortlist, and give the LLM
useful context—not just a larger pile of documents.

### Why retrieve when the model has a huge context window?

**Bring domain evidence beyond the model's training; capacity alone does not select it.**

- **Focus:** retrieve passages relevant to this question, with traceable source IDs.
- **Reuse:** index a corpus once; avoid repeatedly sending every document. Long-context prompt caching can also help.
- **Tradeoff:** retrieval adds complexity and can miss evidence. Sources still need refreshing; answers still need evaluation.

A small, bounded document set may be simpler to put directly in the prompt—even with a million-token window available.

### RAG, tools, and agents are building blocks—not rivals

| Need | Useful starting point |
| --- | --- |
| Read a small, bounded document set | Direct / long-context prompt |
| Repeated questions over a large document collection | RAG over a maintained index |
| Live booking status or an action such as rebooking | Authorized API / tool |
| A multi-step task combining these | Agent using retrieval and tools |

**MCP is a protocol for exposing tools/resources.** An agent can call a RAG search tool through MCP.
Neither MCP nor RAG automatically supplies correct answers or access controls.

### The pipeline: prepare → retrieve → refine → answer

![Full-pipeline diagram: prepare chunks and indexes; rewrite a question, retrieve with BM25 and vectors, fuse ranks, rerank, and generate with optional profile context.](assets/rag-workshop/pipeline.svg)

- **Build / refresh:** legal PDFs → chunks + IDs → lexical index and document vectors.
- **Per question:** parallel search → RRF → cross-encoder → evidence in the LLM prompt.
- **Optional context:** history clarifies the question; saved profile facts personalize the prompt.

*Full-pipeline architecture; standalone demo-wiring differences are noted below.*

## 0 · Setup

In [1]:
import os
import time
import numpy as np
from pathlib import Path
from aerolex import config
from aerolex.loaders import load_documents
from aerolex.embeddings import GeminiEmbedder
from aerolex.generate import Generator
from aerolex.pipeline import RagPipeline
from aerolex.store import InMemoryVectorStore
from aerolex.chunking import chunk_corpus, load_chunks

In [2]:
# Set API key, LLM, Embedding Model 
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
GEN_MODEL = "gemini-3.5-flash-lite"
KEY = config.load_api_key()
CACHE_DIR = r"C:\src\IEEEWorkshop\aerolex-production-rag\cache"

## 1 · Hierarchical Chunking  ·  Feature #1

**Preserve legal identity before worrying about chunk size.**

- Detect **Article / § / RULE** boundaries; keep the heading and source on every part.
- Pack paragraphs within each section; use overlapping character windows when a paragraph is too long.
- Retrieve a passage and still identify it as **“EU 261/2004 — Article 7,”** not merely “chunk 42.”

In [3]:
docs = load_documents()
for d in docs:
    print(f"{d.doc_id:16s} {d.doc_type:6s} {len(d.text):>7,d} chars  {d.title}")

# Prefer the pre-built cache (instant); otherwise chunk live.
try:
    chunks = load_chunks()
    print(f"\nloaded {len(chunks)} chunks from cache")
except FileNotFoundError:
    chunks = chunk_corpus(docs)
    print(f"\nchunked live: {len(chunks)} chunks")

by_id = {c.chunk_id: c for c in chunks}
art7 = next(c for c in chunks if c.section_label == "Article 7")
print("\nExample chunk —", art7.citation)
print(art7.text[:320], "…")

delta_domestic   delta   78,663 chars  Delta Domestic General Rules Tariff (Contract of Carriage)
delta_intl       delta  118,099 chars  Delta International General Rules Tariff (Contract of Carriage)
eu261            eu      27,544 chars  EU Regulation (EC) No 261/2004
cfr260           cfr     26,287 chars  US DOT 14 CFR Part 260

loaded 234 chunks from cache

Example chunk — EU Regulation (EC) No 261/2004 — Article 7
1.
Where reference is made to this Article, passengers shall
receive compensation amounting to:
(a) EUR 250 for all flights of 1 500 kilometres or less;
(b) EUR 400 for all intra-Community flights of more than
1 500 kilometres, and for all other flights between 1 500
and 3 500 kilometres;
(c) EUR 600 for all flights no …


# 3 · Embedding Creation

**An embedding turns searchable text into a reusable numeric representation.**

![Embedding lifecycle: create and save document vectors at index time; embed a new question and compare it with those vectors at query time.](assets/rag-workshop/embedding-cache.svg)

- **Cache document vectors, not answers:** avoid repeating document-embedding requests.
- **Keep IDs aligned:** every vector must belong to the correct chunk; query/document embeddings need compatible spaces.
- **Rebuild together** when text, chunking, model, or dimensions change.

*Demo note: the next cell currently re-embeds on every execution; saving alone does not implement cache reuse.*

In [4]:
if not KEY:
    print("GEMINI_API_KEY not set.")

print(f"Embedding {len(chunks)} chunks with {EMBED_MODEL} "
        f"(dim={EMBED_DIM}) …")

t0 = time.time()
embedder = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM)
matrix = embedder.embed_documents([c.searchable_text for c in chunks])
print(f"Embedded in {time.time() - t0:.1f}s, shape={matrix.shape}")

Embedding 234 chunks with gemini-embedding-001 (dim=768) …


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0, description: Embed content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 3h29m24.899740596s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Embed content requests per day limit exceeded on Free Tier. See https:

In [ ]:
store = InMemoryVectorStore([c.chunk_id for c in chunks], matrix)
store.save(Path(CACHE_DIR) / "new_embeddings.npz")

## 2 · Lexical Search with BM25  ·  Feature #2

**Search by words and identifiers: tokenize → score → rank.**

- **Informative terms matter:** matches on rarer terms usually contribute more.
- **More is not endlessly better:** repeated matches have diminishing gains; document length is accounted for.
- **Token design matters:** keep references such as `260.6` and `261/2004` intact.

BM25 ranks lexical relevance; it is not just a substring or all-words-must-match filter.

### 🛠️ TODO: write `tokenize`

*Exercise scope: this illustrates token design; `BM25Index` currently uses its own library tokenizer.*

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Keep words and dotted/slashed numeric refs as single tokens.
_TOKEN = re.compile(r"[a-z]+|\d+(?:[./-]\d+)*")


def tokenize(text: str) -> list[str]:
    # TODO Fetch list of keywords
    tokens = re.findall(_TOKEN, text.lower())
    return [token for token in tokens if token not in ENGLISH_STOP_WORDS]

assert tokenize("What is the refund for a cancelled flight?") == ["refund", "cancelled", "flight"]
print("tokenize OK")

tokenize OK


In [ ]:
from aerolex.bm25 import BM25Index

bm25 = BM25Index([c.chunk_id for c in chunks], [c.searchable_text for c in chunks])
for cid, score in bm25.search("§ 260.6 refund for a cancelled flight", top_k=3):
    print(f"{score:6.2f}  {by_id[cid].citation}")

 16.02  US DOT 14 CFR Part 260 — § 260.6 (part 6/6)
 15.18  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
 15.00  US DOT 14 CFR Part 260 — § 260.6 (part 3/6)


## 3 · Vector Search + Memory

**Search by learned similarity—not only shared words.**

![Toy comparison: BM25 sees literal cat and street in passage A; vector search can recognize the feline and road paraphrase in B. These are illustrative signals, not measured rankings.](assets/rag-workshop/lexical-vs-dense.svg)

- **Dense retrieval:** embed the question; compare its vector direction with document vectors using cosine similarity.
- **Different strengths:** BM25 anchors identifiers; embeddings can connect paraphrases. Neither guarantees relevance.
- **Hybrid retrieval:** run both, then merge their ranked candidates with RRF.

Here, storage is an **in-memory NumPy matrix**, not conversation history or an external vector database.

### 🛠️ TODO: write `cosine_search`

**Normalize → dot product → sort.** A vector should match itself with cosine ≈ 1.
That checks the search mechanics—not whether the embedding model understands every question.

In [ ]:
def cosine_search(query_vec, matrix, ids, top_k=5):

    # TODO: 1) L2-normalize query_vec and each row of matrix
    q_hat = query_vec / np.sqrt(np.sum(query_vec ** 2))
    matrix_hat = matrix / np.sqrt(np.sum(matrix ** 2, axis=1, keepdims=True))
    #       2) scores = matrix @ query   (cosine, since normalized)
    scores = matrix_hat @ q_hat
    #       3) return the top_k (id, score) pairs, highest first
    top_k = np.argsort(-scores)[:top_k]
    top_chunks = [(ids[i], scores[i]) for i in top_k]
    return top_chunks

# sanity: a row is most similar to itself (cosine ≈ 1.0)
_hit = cosine_search(store.matrix[0], store.matrix, store.ids, top_k=1)[0]
print(_hit[0] == store.ids[0], round(_hit[1], 4))

True 1.0


In [ ]:
# Embed a real query (1 API call) and search. Needs a key; else we reuse a chunk vector.
query = "What is the refund for a cancelled flight"

query_vector = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM).embed_query(query)

dense_hits = cosine_search(query_vector, store.matrix, store.ids, top_k=5)
for cid, s in dense_hits:
    print(f"{s:5.3f}  {by_id[cid].citation}")

0.726  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
0.724  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
0.713  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
0.698  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)
0.696  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 5/6)


## 4 · Reciprocal Rank Fusion  ·  Feature #5

**Let each ranked list vote; do not add BM25 scores to cosine scores.**

![RRF example: lexical ranks A B C and vector ranks B C A become rank-based votes; B has the largest combined vote.](assets/rag-workshop/rank-fusion.svg)

- **Rank means position:** first, second, third—not the original score magnitude.
- **A high position earns a bigger vote;** appearing high in multiple lists helps.
- **Missing from a list?** No vote from that list. A candidate can still enter through either retriever.

### Why reciprocal rank? A small, decreasing vote

$$
\mathrm{RRF}(d) = \sum_{L: d \in L} \frac{1}{60 + \mathrm{rank}_L(d)}
$$

| Passage | BM25 rank | Vector rank | Added votes | Total |
| --- | ---: | ---: | --- | ---: |
| B | 2 | 1 | 1/62 + 1/61 | **0.03252** |
| A | 1 | 3 | 1/61 + 1/63 | 0.03227 |
| C | 3 | 2 | 1/63 + 1/62 | 0.03200 |

**Intuition:** reciprocal decay still credits lower ranks; adding `k` softens an outlier's first-place advantage.
This is an established heuristic: the original RRF paper selected **60** in pilot experiments—not a universal optimum.

Ranks here start at **1**; the code uses `rank + 1` because `enumerate` starts at 0.
`k` is smoothing, **not** `top_k` (how many results to keep). These totals are not probabilities.

### 🛠️ TODO: write `rrf`

In [ ]:
def rrf(rankings, k=60, top_k=10):
    scores = {}
    for ranked in rankings:
        for rank, cid in enumerate(ranked):
            chunk_score = scores.get(cid, 0.0)
            # TODO for each chunk compute the RRF Score
            scores[cid] = chunk_score + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:top_k]

assert rrf([["b", "a", "c"], ["b", "a", "d"]])[0][0] == "b"
print("rrf OK")

rrf OK


In [ ]:
bm25_ids = [cid for cid, _ in bm25.search(query, top_k=20)]
dense_ids = [cid for cid, _ in dense_hits]
fused = rrf([bm25_ids, dense_ids], k=config.RRF_K, top_k=10)
print("Fused top-5:")
for cid, s in fused[:5]:
    print(f"{s:.4f}  {by_id[cid].citation}")

Fused top-5:
0.0323  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
0.0303  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
0.0296  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
0.0161  US DOT 14 CFR Part 260 — § 260.1
0.0159  US DOT 14 CFR Part 260 — § 260.6 (part 1/6)


## 5 · Cross-Encoder Re-Ranking  ·  Feature #6

**Retrieval finds candidates. A joint reader refines their order.**

![Cross-encoder: each fused candidate is paired with the question, jointly scored, and reordered before selecting LLM context. Toy scores are not probabilities.](assets/rag-workshop/cross-encoder.svg)

- **Unlike cached document embeddings,** each score needs a fresh read of question + passage together.
- **Spend that work on a shortlist,** not the whole corpus; finer relevance judgments can improve ordering.
- **Limits:** scores are not probabilities; reranking cannot recover missed candidates or verify the final answer.

*The cross-encoder runs locally; its public model may download on first use.*

In [ ]:
from aerolex.rerank import CrossEncoderReranker

candidates = [(cid, by_id[cid].searchable_text) for cid, _ in fused]
reranked = CrossEncoderReranker().rerank(query, candidates, top_k=config.RERANK_TOP_K)
print("Re-ranked top chunks:")
for cid, s in reranked:
    print(f"{s:7.2f}  {by_id[cid].citation}")
final_chunks = [by_id[cid] for cid, _ in reranked]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Re-ranked top chunks:
   6.68  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
   6.28  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
   5.77  US DOT 14 CFR Part 260 — § 260.6 (part 1/6)
   5.55  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
   5.28  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
   5.24  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
   3.78  US DOT 14 CFR Part 260 — § 260.1
   1.71  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 5/6)
   0.84  Delta International General Rules Tariff (Contract of Carriage) — RULE 15 (part 4/4)
  -0.09  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)


## 6 . Generate Answer

**Turn supplied evidence into an answer—not an automatic proof.**

- Give the model the question, context passages, and a clear instruction to acknowledge missing evidence.
- Request source tags such as `[[chunk_id]]`; the current generation path returns text **without citation validation**.

*Demo-wiring note: the standalone call below passes all `chunks`, not `final_chunks`.
The full `RagPipeline` uses retrieved/reranked context. Code alignment and rerunning are a separate pass.*

In [ ]:
# TODO: Write a system prompt for the AeroLex assistant
system_prompt = (
    "You are AeroLex, an assistant answering questions about airline passenger "
    "rights using ONLY the provided context passages. Every factual sentence "
    "must end with a citation tag like [[chunk_id]] using the exact chunk_id of the passage "
    "you used. If the context does not answer the question, say so plainly. "
    "Never invent regulations, article numbers, or passage numbers."
)

In [ ]:
user_prompt_template = (
    "{profile_block}\n\n"
    "Context passages:\n{context_text}\n\n"
    "Question: {query}\n\n"
    "Answer (cite every factual claim with [[chunk_id]]):"
)

result = Generator(api_key = KEY, model=GEN_MODEL).generate(
    query, chunks,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
)

print(result)

When a flight is cancelled, the refund provisions depend on the specific rules and regulations governing your flight:

* **Delta Domestic Flights:** If a flight is cancelled or significantly changed (such as a departure or arrival time change of 180 minutes or more), Delta will cancel the remaining ticket and provide a full refund of the unused portion of the ticket and unused ancillary fees (including paid checked bag fees) in the original form of payment upon request, provided you do not accept an alternative flight or voucher option or fail to respond prior to the originally scheduled departure time [[delta_domestic::rule-19::0], [delta_domestic::rule-22::0]]. If no portion of the ticket has been used, the refund equals the full fare paid [[delta_domestic::rule-22::0]]. For fully refundable tickets, if no portion has been used, the total fare and all taxes, fees, and charges paid will be refunded or made available for future travel [[delta_domestic::rule-22::3]].
* **Delta Internati

## 7 · Conversational Memory 

**Make the question self-contained before searching.**

> “What about in the US?” → “US refund rules for cancelled flights?” *(illustrative rewrite)*

History supplies the missing subject; query reformulation carries it into retrieval.
This is conversation context—not the document-vector store.

In [ ]:
from aerolex.memory import ConversationMemory

mem = ConversationMemory()
mem.add_user(query)
mem.add_assistant(result)
print("follow-up : what about in the US?")
print("standalone:", mem.condense("what about in the US?"))

follow-up : what about in the US?
standalone: US Department of Transportation refund regulations for cancelled flights


## 8 · Long-Term Persistence 

**Remember selected user facts across sessions.**

- SQLite stores facts such as a home airport; the profile can be included in later prompts.
- This demo **explicitly saves** facts—it does not automatically extract every fact from conversation.
- Profile context personalizes an answer; it is not evidence for a legal entitlement.

In [ ]:
from aerolex.persistence import ProfileStore

prof = ProfileStore()
prof.remember("demo_user", "home_airport", "JFK")
prof.remember("demo_user", "status", "Delta Platinum Medallion")
print(prof.context_block("demo_user"))

Known facts about the user:
- home_airport: JFK
- status: Delta Platinum Medallion


## 10 · The Full Pipeline

**Connect the stages into one reusable path.**

**Question → rewrite → BM25 + vector → RRF → rerank → context + profile → answer.**

The library composes the pieces; it does not remove retrieval errors or guarantee factual support.
Saved outputs and errors below are retained from an earlier run; execution cleanup comes after this narrative pass.

In [ ]:
pipe = RagPipeline(
    chunks=chunks,
    store=store,
    bm25=bm25,
    embedder=GeminiEmbedder(),
    reranker=CrossEncoderReranker(),
    generator=Generator(),
)
out = pipe.answer("If my 4000 km flight is cancelled, what am I owed?",
                    memory=mem, profile=prof, user_id="demo_user",
                    system_prompt=system_prompt,
                    user_prompt_template=user_prompt_template)
print(out["answer"])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

AttributeError: 'str' object has no attribute 'update'

### Optional reading

- **Tool access vs application logic:** [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture).
- **Where the rank-voting rule comes from:** [Cormack, Clarke & Büttcher — Reciprocal Rank Fusion, SIGIR 2009](https://cormack.uwaterloo.ca/cormacksigir09-rrf.pdf).

*The toy retrieval preferences, RRF lists, and reranker scores are teaching illustrations—not measured results for this notebook.*